# Notebook 02 — Data Acquisition & Initial Inspection
## Section 1: Load Monthly DB Data & Filter ICE (Multi-Month)

**Why this section**  
We download Deutsche Bahn monthly Parquet files and keep **ICE only**, then save locally so later notebooks never re-download.

**Months**  
From `project_config.json`: **2024-07, 2024-08, 2024-09**

**Approach**  
For each month:  
1. `hf_hub_download` monthly file (cached after first run)  
2. Chunked read + filter `train_type == "ICE"`  
3. Save `ice_raw_YYYY-MM.parquet` + load report JSON  

**Why chunked?** Monthly files contain all train types; chunking keeps memory low.

**Outputs (per month)**  
- `data/processed/ice_raw_YYYY-MM.parquet`  
- `data/reference/ice_load_report_YYYY-MM.json`  
- Plus summary: `data/reference/ice_load_summary_multi_month.json`

In [1]:
# =============================================================================
# Notebook 02 | Section 1: Load DB Monthly Data & Filter ICE (Multi-Month)
# =============================================================================
# Self-contained: loads project_config; loops TARGET_MONTHS;
# downloads each month, filters ICE, saves parquet + reports.
# =============================================================================

from __future__ import annotations

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError(
        "Cannot find data/reference/project_config.json. Run Notebook 01 first."
    )


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

ICE_FILTER = "ICE"
CHUNK_SIZE = 100_000
# If True: skip months that already have ice_raw_YYYY-MM.parquet
SKIP_IF_EXISTS = True


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}\nRun Notebook 01 first.")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            pass
    return obj


config = load_json(CONFIG_PATH)
TARGET_MONTHS = config["scope"]["target_months"]
HF_DATASET_ID = config["data_sources"]["deutsche_bahn"]["hf_dataset_id"]
VERIFIED_COLUMNS = [c["name"] for c in config["db_verified_columns"]]
PARQUET_PATTERN = config["data_sources"]["deutsche_bahn"]["monthly_parquet_pattern"]
PRIMARY_TARGET = config["ml_tasks"]["primary"]["target"]


def require_package(import_name: str, pip_name: str) -> None:
    try:
        __import__(import_name)
    except ImportError as exc:
        raise ImportError(
            f"Package '{import_name}' not installed.\nRun: pip install {pip_name}"
        ) from exc


require_package("huggingface_hub", "huggingface_hub")
from huggingface_hub import hf_hub_download


def load_ice_chunked(
    parquet_path: str,
    columns: list[str],
    train_type: str = ICE_FILTER,
    chunk_size: int = CHUNK_SIZE,
    month_label: str = "",
) -> tuple[pd.DataFrame, int]:
    parquet_file = pq.ParquetFile(parquet_path)
    ice_chunks: list[pd.DataFrame] = []
    total_rows_scanned = 0

    for batch in parquet_file.iter_batches(batch_size=chunk_size, columns=columns):
        chunk = batch.to_pandas()
        total_rows_scanned += len(chunk)
        ice_chunk = chunk[chunk["train_type"] == train_type]
        if not ice_chunk.empty:
            ice_chunks.append(ice_chunk)

        if total_rows_scanned % 500_000 < chunk_size:
            ice_so_far = sum(len(c) for c in ice_chunks)
            print(
                f"  [{month_label}] Scanned {total_rows_scanned:>10,} | "
                f"ICE found: {ice_so_far:>8,}"
            )

    if not ice_chunks:
        raise ValueError(
            f"No ICE rows found for {month_label}. Check month file and filter."
        )

    return pd.concat(ice_chunks, ignore_index=True), total_rows_scanned


def process_month(target_month: str) -> dict:
    """Download one month, filter ICE, save parquet + report. Returns summary dict."""
    output_parquet = PROCESSED_DIR / f"ice_raw_{target_month}.parquet"
    load_report_path = REFERENCE_DIR / f"ice_load_report_{target_month}.json"
    parquet_filename = PARQUET_PATTERN.format(year_month=target_month)

    print("=" * 72)
    print(f"MONTH: {target_month}")
    print("=" * 72)

    if SKIP_IF_EXISTS and output_parquet.exists():
        print(f"SKIP: {output_parquet.name} already exists (SKIP_IF_EXISTS=True)")
        existing = pd.read_parquet(output_parquet)
        summary = {
            "month": target_month,
            "status": "skipped_existing",
            "ice_rows": int(len(existing)),
            "output_parquet": str(output_parquet),
        }
        print(f"  Existing ICE rows: {len(existing):,}")
        print()
        return summary

    print(f"Downloading: {HF_DATASET_ID}/{parquet_filename}")
    download_start = time.time()
    local_parquet_path = hf_hub_download(
        repo_id=HF_DATASET_ID,
        filename=parquet_filename,
        repo_type="dataset",
    )
    download_seconds = round(time.time() - download_start, 1)
    print(f"Download/cache complete in {download_seconds}s")
    print(f"Local path: {local_parquet_path}")
    print()

    print(f"Filtering train_type == '{ICE_FILTER}' (chunked)...")
    filter_start = time.time()
    ice_df, total_rows_scanned = load_ice_chunked(
        parquet_path=local_parquet_path,
        columns=VERIFIED_COLUMNS,
        month_label=target_month,
    )
    filter_seconds = round(time.time() - filter_start, 1)
    ice_pct = round(100 * len(ice_df) / total_rows_scanned, 2)
    print(f"Filtering complete in {filter_seconds}s")
    print(f"  Total scanned : {total_rows_scanned:,}")
    print(f"  ICE kept      : {len(ice_df):,} ({ice_pct}% of month)")
    print()

    missing_cols = [c for c in VERIFIED_COLUMNS if c not in ice_df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns for {target_month}: {missing_cols}")

    non_ice = ice_df[ice_df["train_type"] != ICE_FILTER]
    if len(non_ice) > 0:
        raise ValueError(f"Filter failed for {target_month}: {len(non_ice)} non-ICE rows")

    timestamp_cols = [
        "time",
        "arrival_planned_time",
        "arrival_change_time",
        "departure_planned_time",
        "departure_change_time",
    ]
    for col in timestamp_cols:
        ice_df[col] = pd.to_datetime(ice_df[col], errors="coerce")

    delay_stats = {
        "min": int(ice_df["delay_in_min"].min()),
        "max": int(ice_df["delay_in_min"].max()),
        "mean": round(float(ice_df["delay_in_min"].mean()), 2),
        "median": float(ice_df["delay_in_min"].median()),
        "early_count": int((ice_df["delay_in_min"] < 0).sum()),
        "on_time_count": int((ice_df["delay_in_min"] == 0).sum()),
        "late_count": int((ice_df["delay_in_min"] > 0).sum()),
    }

    canceled_stops = int(ice_df["is_canceled"].sum())
    inspection_summary = {
        "rows": int(len(ice_df)),
        "columns": int(len(ice_df.columns)),
        "unique_stations_eva": int(ice_df["eva"].nunique()),
        "unique_train_numbers": int(ice_df["train_number"].nunique()),
        "unique_rides": int(ice_df["train_line_ride_id"].nunique()),
        "canceled_stops": canceled_stops,
        "canceled_pct": round(100 * canceled_stops / len(ice_df), 2),
        "date_range_time_col": {
            "min": str(ice_df["time"].min()),
            "max": str(ice_df["time"].max()),
        },
        "delay_stats": delay_stats,
    }

    ice_df.to_parquet(output_parquet, index=False)

    load_report = {
        "metadata": {
            "notebook": "02_Data_Acquisition_and_Initial_Inspection",
            "section": "Section 1",
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "target_month": target_month,
            "hf_dataset_id": HF_DATASET_ID,
            "source_file": parquet_filename,
            "local_parquet_cache": str(local_parquet_path),
            "primary_target": PRIMARY_TARGET,
            "primary_metric": "mae",
        },
        "processing": {
            "ice_filter": ICE_FILTER,
            "chunk_size": CHUNK_SIZE,
            "total_rows_scanned": int(total_rows_scanned),
            "ice_rows_saved": int(len(ice_df)),
            "ice_pct_of_month": ice_pct,
            "download_seconds": download_seconds,
            "filter_seconds": filter_seconds,
        },
        "output_files": {
            "ice_parquet": str(output_parquet),
            "load_report": str(load_report_path),
        },
        "inspection_summary": inspection_summary,
        "columns": VERIFIED_COLUMNS,
    }

    with load_report_path.open("w", encoding="utf-8") as f:
        json.dump(make_json_safe(load_report), f, indent=2, ensure_ascii=False)

    print(f"Saved: {output_parquet}")
    print(f"Report: {load_report_path}")
    print(f"  Stations: {inspection_summary['unique_stations_eva']}")
    print(f"  Delay mean/median: {delay_stats['mean']} / {delay_stats['median']} min")
    print()

    return {
        "month": target_month,
        "status": "saved",
        "ice_rows": int(len(ice_df)),
        "rows_scanned": int(total_rows_scanned),
        "ice_pct": ice_pct,
        "unique_stations_eva": inspection_summary["unique_stations_eva"],
        "delay_mean": delay_stats["mean"],
        "delay_median": delay_stats["median"],
        "output_parquet": str(output_parquet),
        "load_report": str(load_report_path),
    }


# =============================================================================
# RUN ALL TARGET MONTHS
# =============================================================================
print("Notebook 02 | Section 1 — Multi-month ICE acquisition")
print(f"Months: {', '.join(TARGET_MONTHS)}")
print(f"Target: {PRIMARY_TARGET} | Metric: MAE")
print(f"SKIP_IF_EXISTS = {SKIP_IF_EXISTS}")
print()

month_results = []
for month in TARGET_MONTHS:
    month_results.append(process_month(month))

summary_path = REFERENCE_DIR / "ice_load_summary_multi_month.json"
summary = {
    "metadata": {
        "notebook": "02_Data_Acquisition_and_Initial_Inspection",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
    },
    "months": month_results,
    "totals": {
        "ice_rows_all_months": int(sum(r["ice_rows"] for r in month_results)),
        "months_processed": len(month_results),
    },
}

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 1 COMPLETE — Multi-month summary")
print(sep)
print(f"{'Month':<10} {'Status':<18} {'ICE rows':>12}")
for r in month_results:
    print(f"{r['month']:<10} {r['status']:<18} {r['ice_rows']:>12,}")
print()
print(f"Total ICE rows : {summary['totals']['ice_rows_all_months']:,}")
print(f"Summary saved  : {summary_path}")
print(sep)
print("Next: Section 2 — profile ICE data across months")
print(sep)

Notebook 02 | Section 1 — Multi-month ICE acquisition
Months: 2024-07, 2024-08, 2024-09
Target: delay_in_min | Metric: MAE
SKIP_IF_EXISTS = True

MONTH: 2024-07
SKIP: ice_raw_2024-07.parquet already exists (SKIP_IF_EXISTS=True)
  Existing ICE rows: 157,886

MONTH: 2024-08
Downloading: piebro/deutsche-bahn-data/monthly_processed_data/data-2024-08.parquet


monthly_processed_data/data-2024-08.parq(…):   0%|          | 0.00/100M [00:00<?, ?B/s]

c:\Users\Manikanta\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Manikanta\.cache\huggingface\hub\datasets--piebro--deutsche-bahn-data. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Download/cache complete in 23.0s
Local path: C:\Users\Manikanta\.cache\huggingface\hub\datasets--piebro--deutsche-bahn-data\snapshots\d1bcb820f41454eafe221b8cd91cfe3fee9b5f30\monthly_processed_data\data-2024-08.parquet

Filtering train_type == 'ICE' (chunked)...
  [2024-08] Scanned    500,000 | ICE found:   37,719
  [2024-08] Scanned  1,000,000 | ICE found:   77,949
  [2024-08] Scanned  1,500,000 | ICE found:  119,254
Filtering complete in 2.1s
  Total scanned : 1,888,035
  ICE kept      : 150,289 (7.96% of month)

Saved: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_raw_2024-08.parquet
Report: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\ice_load_report_2024-08.json
  Stations: 98
  Delay mean/median: 11.91 / 4.0 min

MONTH: 2024-09
Downloading: piebro/deutsche-bahn-data/monthly_processed_data/data-2024-09.parquet


monthly_processed_data/data-2024-09.parq(…):   0%|          | 0.00/101M [00:00<?, ?B/s]

Download/cache complete in 18.2s
Local path: C:\Users\Manikanta\.cache\huggingface\hub\datasets--piebro--deutsche-bahn-data\snapshots\d1bcb820f41454eafe221b8cd91cfe3fee9b5f30\monthly_processed_data\data-2024-09.parquet

Filtering train_type == 'ICE' (chunked)...
  [2024-09] Scanned    500,000 | ICE found:   39,349
  [2024-09] Scanned  1,000,000 | ICE found:   78,673
  [2024-09] Scanned  1,500,000 | ICE found:  116,866
Filtering complete in 2.4s
  Total scanned : 1,904,404
  ICE kept      : 149,218 (7.84% of month)

Saved: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_raw_2024-09.parquet
Report: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\ice_load_report_2024-09.json
  Stations: 98
  Delay mean/median: 10.86 / 3.0 min

Section 1 COMPLETE — Multi-month summary
Month      Status                 ICE rows
2024-07    skipped_existing        157,886
2024-08    saved                   150,289
2024-09    saved                   149,218

Total ICE rows : 4

# Section 2: ICE Dataset Profiling (Multi-Month)

**Why this section**  
Before cleaning, we profile each saved ICE month: nulls, cancellations, stations, and **`delay_in_min`** (regression target).

**Input**  
`data/processed/ice_raw_YYYY-MM.parquet` for months in `project_config.json`

**What we check**  
- Column null rates  
- Cancellation rate  
- Delay distribution (mean / median / percentiles) — relevant for **MAE** later  
- Unique stations and trains  

**Outputs**  
- Per month: `ice_profile_report_YYYY-MM.json`  
- Summary: `ice_profile_summary_multi_month.json`

In [2]:
# =============================================================================
# Notebook 02 | Section 2: ICE Dataset Profiling (Multi-Month)
# =============================================================================
# Self-contained: loads ice_raw_*.parquet from disk for each target month.
# Saves per-month profile reports + multi-month summary.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import numpy as np


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError(
        "Cannot find project_config.json. Run Notebook 01 and NB02 Section 1 first."
    )


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


config = load_json(CONFIG_PATH)
TARGET_MONTHS = config["scope"]["target_months"]
VERIFIED_COLUMNS = [c["name"] for c in config["db_verified_columns"]]
ICE_FILTER = config["scope"]["train_type_filter"]
PRIMARY_TARGET = config["ml_tasks"]["primary"]["target"]

TIMESTAMP_COLS = [
    "time",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
]


def profile_columns(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        rows.append(
            {
                "column": col,
                "dtype": str(df[col].dtype),
                "null_count": int(df[col].isna().sum()),
                "null_pct": round(100 * df[col].isna().mean(), 2),
                "n_unique": int(df[col].nunique(dropna=True)),
            }
        )
    return (
        pd.DataFrame(rows)
        .sort_values("null_pct", ascending=False)
        .reset_index(drop=True)
    )


def profile_month(target_month: str) -> dict:
    ice_parquet = PROCESSED_DIR / f"ice_raw_{target_month}.parquet"
    profile_path = REFERENCE_DIR / f"ice_profile_report_{target_month}.json"
    load_report_path = REFERENCE_DIR / f"ice_load_report_{target_month}.json"

    if not ice_parquet.exists():
        raise FileNotFoundError(
            f"Missing: {ice_parquet}\nRun Notebook 02 Section 1 first."
        )

    print("=" * 72)
    print(f"PROFILING MONTH: {target_month}")
    print("=" * 72)
    print(f"Loading: {ice_parquet}")
    ice_df = pd.read_parquet(ice_parquet)

    if load_report_path.exists():
        load_report = load_json(load_report_path)
        expected_rows = load_report["processing"]["ice_rows_saved"]
        if len(ice_df) != expected_rows:
            print(
                f"WARNING: row mismatch — Section 1 saved {expected_rows:,}, "
                f"loaded {len(ice_df):,}"
            )

    missing_cols = [c for c in VERIFIED_COLUMNS if c not in ice_df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")

    ice_df = ice_df[VERIFIED_COLUMNS]
    non_ice = ice_df[ice_df["train_type"] != ICE_FILTER]
    if len(non_ice) > 0:
        raise ValueError(f"Found {len(non_ice)} non-ICE rows in {target_month}")

    for col in TIMESTAMP_COLS:
        ice_df[col] = pd.to_datetime(ice_df[col], errors="coerce")

    column_profile = profile_columns(ice_df)
    delay = ice_df["delay_in_min"]

    delay_distribution = {
        "count": int(len(delay)),
        "min": int(delay.min()),
        "max": int(delay.max()),
        "mean": round(float(delay.mean()), 2),
        "median": float(delay.median()),
        "std": round(float(delay.std()), 2),
        "p25": float(delay.quantile(0.25)),
        "p75": float(delay.quantile(0.75)),
        "p90": float(delay.quantile(0.90)),
        "p95": float(delay.quantile(0.95)),
        "early_arrivals_lt_0": int((delay < 0).sum()),
        "on_time_eq_0": int((delay == 0).sum()),
        "late_gt_0": int((delay > 0).sum()),
        "note": (
            "delay_in_min is the regression target; "
            "MAE will measure average absolute error in minutes."
        ),
    }

    canceled = ice_df["is_canceled"]
    cancellation = {
        "canceled_count": int(canceled.sum()),
        "canceled_pct": round(100 * float(canceled.mean()), 2),
        "action": "Drop canceled stops in Notebook 03 (no valid delay outcome).",
    }

    station_summary = {
        "unique_eva_stations": int(ice_df["eva"].nunique()),
        "unique_station_name": int(ice_df["station_name"].nunique(dropna=True)),
        "unique_xml_station_name": int(ice_df["xml_station_name"].nunique(dropna=True)),
        "station_name_null_pct": round(100 * ice_df["station_name"].isna().mean(), 2),
    }

    train_summary = {
        "unique_train_numbers": int(ice_df["train_number"].nunique()),
        "unique_rides": int(ice_df["train_line_ride_id"].nunique()),
        "line_number_null_pct": round(100 * ice_df["line_number"].isna().mean(), 2),
        "note": "line_number often 100% null for ICE — expected.",
    }

    time_range = {
        "time_min": str(ice_df["time"].min()),
        "time_max": str(ice_df["time"].max()),
        "departure_planned_null_pct": round(
            100 * ice_df["departure_planned_time"].isna().mean(), 2
        ),
    }

    high_null = column_profile[column_profile["null_pct"] > 10]["column"].tolist()
    quality_flags = {
        "high_null_columns_gt_10pct": high_null,
        "only_ice": True,
        "has_negative_delays": bool((delay < 0).any()),
        "has_cancellations": bool(canceled.any()),
        "implications": [
            "Cancelations removed in Notebook 03.",
            "Negative delays kept for regression (early arrivals).",
            "Missing departure_planned_time → non-mergeable for weather.",
            "line_number nulls kept (do not drop ICE rows for this).",
        ],
    }

    profile_report = {
        "metadata": {
            "notebook": "02_Data_Acquisition_and_Initial_Inspection",
            "section": "Section 2",
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "target_month": target_month,
            "source_parquet": str(ice_parquet),
            "primary_target": PRIMARY_TARGET,
            "primary_metric": "mae",
        },
        "dataset_shape": {"rows": int(len(ice_df)), "columns": int(len(ice_df.columns))},
        "column_profile": column_profile.to_dict(orient="records"),
        "delay_distribution": delay_distribution,
        "cancellation": cancellation,
        "station_summary": station_summary,
        "train_summary": train_summary,
        "time_range": time_range,
        "quality_flags": quality_flags,
    }

    with profile_path.open("w", encoding="utf-8") as f:
        json.dump(make_json_safe(profile_report), f, indent=2, ensure_ascii=False)

    print(f"Rows          : {len(ice_df):,}")
    print(f"Stations (eva): {station_summary['unique_eva_stations']}")
    print(f"Canceled      : {cancellation['canceled_pct']}%")
    print(
        f"Delay mean/median: {delay_distribution['mean']} / "
        f"{delay_distribution['median']} min"
    )
    print(f"Saved report  : {profile_path}")
    print()
    print("Top null columns:")
    print(column_profile.head(6).to_string(index=False))
    print()

    return {
        "month": target_month,
        "rows": int(len(ice_df)),
        "unique_eva_stations": station_summary["unique_eva_stations"],
        "canceled_pct": cancellation["canceled_pct"],
        "delay_mean": delay_distribution["mean"],
        "delay_median": delay_distribution["median"],
        "departure_planned_null_pct": time_range["departure_planned_null_pct"],
        "profile_report": str(profile_path),
    }


# =============================================================================
# RUN ALL MONTHS
# =============================================================================
print("Notebook 02 | Section 2 — Multi-month ICE profiling")
print(f"Months : {', '.join(TARGET_MONTHS)}")
print(f"Target : {PRIMARY_TARGET} | Metric: MAE")
print()

month_summaries = [profile_month(m) for m in TARGET_MONTHS]

summary_path = REFERENCE_DIR / "ice_profile_summary_multi_month.json"
summary = {
    "metadata": {
        "notebook": "02_Data_Acquisition_and_Initial_Inspection",
        "section": "Section 2",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
    },
    "months": month_summaries,
    "totals": {
        "ice_rows_all_months": int(sum(m["rows"] for m in month_summaries)),
        "mean_delay_by_month": {
            m["month"]: m["delay_mean"] for m in month_summaries
        },
    },
}

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 2 COMPLETE — Multi-month profile summary")
print(sep)
print(
    f"{'Month':<10} {'Rows':>10} {'Stations':>10} "
    f"{'Cancel%':>8} {'Mean delay':>11} {'Median':>8}"
)
for m in month_summaries:
    print(
        f"{m['month']:<10} {m['rows']:>10,} {m['unique_eva_stations']:>10} "
        f"{m['canceled_pct']:>8.2f} {m['delay_mean']:>11.2f} {m['delay_median']:>8.1f}"
    )
print()
print(f"Total ICE rows : {summary['totals']['ice_rows_all_months']:,}")
print(f"Summary saved  : {summary_path}")
print(sep)
print("Next: Section 3 — Open-Meteo weather sample check (structure)")
print(sep)

Notebook 02 | Section 2 — Multi-month ICE profiling
Months : 2024-07, 2024-08, 2024-09
Target : delay_in_min | Metric: MAE

PROFILING MONTH: 2024-07
Loading: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_raw_2024-07.parquet
Rows          : 157,886
Stations (eva): 95
Canceled      : 7.28%
Delay mean/median: 10.9 / 4.0 min
Saved report  : c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\ice_profile_report_2024-07.json

Top null columns:
                column          dtype  null_count  null_pct  n_unique
           line_number         object      157886    100.00         0
departure_planned_time datetime64[ns]       15452      9.79     37610
 departure_change_time datetime64[ns]       15441      9.78     38066
  arrival_planned_time datetime64[ns]       15310      9.70     37765
   arrival_change_time datetime64[ns]       15307      9.69     38099
          station_name         object           0      0.00        91

PROFILING MONTH: 2024-08
Loading: c

# Section 3: Open-Meteo Weather Sample & Time Alignment (Multi-Month)

**Why this section**  
Confirm weather API works for each project month and that railway minutes can be floored to hours for joining.

**What we do (per month)**  
1. Load `ice_raw_YYYY-MM.parquet`  
2. Fetch demo Berlin weather for that month’s date range  
3. Show minute → hour flooring for merge  
4. Save sample parquet + inspection report  

**Important**  
This is a **structure check**, not the final weather dataset. Notebook 04 geocodes every `eva` and fetches real station weather.

**Outputs**  
- `weather_sample_YYYY-MM.parquet`  
- `weather_inspection_report_YYYY-MM.json`  
- `weather_inspection_summary_multi_month.json`

In [3]:
# =============================================================================
# Notebook 02 | Section 3: Open-Meteo Weather Sample & Time Alignment
# =============================================================================
# Self-contained: loads ICE parquet per month; demo Open-Meteo fetch;
# checks hour alignment; saves sample + reports.
# =============================================================================

from __future__ import annotations

import json
from calendar import monthrange
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"
WEATHER_DICT_PATH = REFERENCE_DIR / "weather_data_dictionary.json"

OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
REQUEST_TIMEOUT = 60

DEMO_LATITUDE = 52.525
DEMO_LONGITUDE = 13.369
DEMO_TIMEZONE = "Europe/Berlin"

HOURLY_VARIABLES = [
    "temperature_2m",
    "precipitation",
    "rain",
    "snowfall",
    "windspeed_10m",
    "windgusts_10m",
    "weather_code",
    "visibility",
]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


config = load_json(CONFIG_PATH)
weather_dict = load_json(WEATHER_DICT_PATH)
TARGET_MONTHS = config["scope"]["target_months"]
PRIMARY_TARGET = config["ml_tasks"]["primary"]["target"]


def fetch_open_meteo_archive(
    latitude: float,
    longitude: float,
    start_date: str,
    end_date: str,
    hourly_variables: list[str],
    timezone: str = "Europe/Berlin",
) -> dict:
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_variables),
        "timezone": timezone,
    }
    response = requests.get(
        OPEN_METEO_ARCHIVE_URL, params=params, timeout=REQUEST_TIMEOUT
    )
    response.raise_for_status()
    return response.json()


def parse_hourly_response(response: dict) -> pd.DataFrame:
    hourly = response["hourly"]
    df = pd.DataFrame(hourly)
    df["time"] = pd.to_datetime(df["time"])
    df["latitude"] = response.get("latitude")
    df["longitude"] = response.get("longitude")
    df["timezone"] = response.get("timezone")
    return df


def floor_to_hour(dt_series: pd.Series, tz: str = "Europe/Berlin") -> pd.Series:
    """Floor timestamps to hour for weather merge key."""
    if dt_series.dt.tz is None:
        localized = dt_series.dt.tz_localize(tz, ambiguous="NaT", nonexistent="NaT")
    else:
        localized = dt_series.dt.tz_convert(tz)
    return localized.dt.floor("h").dt.tz_localize(None)


def month_date_bounds(target_month: str) -> tuple[str, str]:
    """Return YYYY-MM-DD start/end for a calendar month."""
    year, month = map(int, target_month.split("-"))
    last_day = monthrange(year, month)[1]
    return f"{target_month}-01", f"{target_month}-{last_day:02d}"


def process_month_weather(target_month: str) -> dict:
    ice_parquet = PROCESSED_DIR / f"ice_raw_{target_month}.parquet"
    weather_parquet = PROCESSED_DIR / f"weather_sample_{target_month}.parquet"
    report_path = REFERENCE_DIR / f"weather_inspection_report_{target_month}.json"

    if not ice_parquet.exists():
        raise FileNotFoundError(
            f"Missing: {ice_parquet}\nRun Notebook 02 Section 1 first."
        )

    print("=" * 72)
    print(f"WEATHER SAMPLE — {target_month}")
    print("=" * 72)

    ice_df = pd.read_parquet(ice_parquet)
    for col in [
        "time",
        "arrival_planned_time",
        "arrival_change_time",
        "departure_planned_time",
        "departure_change_time",
    ]:
        ice_df[col] = pd.to_datetime(ice_df[col], errors="coerce")

    # Prefer ICE date range; fall back to calendar month bounds
    if ice_df["time"].notna().any():
        start_date = str(ice_df["time"].min().date())
        end_date = str(ice_df["time"].max().date())
    else:
        start_date, end_date = month_date_bounds(target_month)

    print(f"ICE rows      : {len(ice_df):,}")
    print(f"Unique eva    : {ice_df['eva'].nunique()}")
    print(f"Date range    : {start_date} → {end_date}")
    print(f"Demo coords   : ({DEMO_LATITUDE}, {DEMO_LONGITUDE}) Berlin")
    print("Fetching Open-Meteo...")

    api_response = fetch_open_meteo_archive(
        latitude=DEMO_LATITUDE,
        longitude=DEMO_LONGITUDE,
        start_date=start_date,
        end_date=end_date,
        hourly_variables=HOURLY_VARIABLES,
        timezone=DEMO_TIMEZONE,
    )

    weather_df = parse_hourly_response(api_response)
    missing_vars = [v for v in HOURLY_VARIABLES if v not in weather_df.columns]
    if missing_vars:
        raise ValueError(f"API missing variables: {missing_vars}")

    weather_df.to_parquet(weather_parquet, index=False)

    ice_with_dep = ice_df[ice_df["departure_planned_time"].notna()].copy()
    ice_with_dep["departure_planned_hour"] = floor_to_hour(
        ice_with_dep["departure_planned_time"]
    )

    alignment_sample = (
        ice_with_dep[
            [
                "eva",
                "xml_station_name",
                "departure_planned_time",
                "departure_planned_hour",
                "delay_in_min",
            ]
        ]
        .head(5)
        .copy()
    )
    for c in ["departure_planned_time", "departure_planned_hour"]:
        alignment_sample[c] = alignment_sample[c].astype(str)

    expected_hours = (
        pd.Timestamp(end_date) - pd.Timestamp(start_date)
    ).days + 1
    expected_hours *= 24

    weather_summary = {
        "rows": int(len(weather_df)),
        "expected_hours_approx": int(expected_hours),
        "date_min": str(weather_df["time"].min()),
        "date_max": str(weather_df["time"].max()),
        "temperature_2m_mean": round(float(weather_df["temperature_2m"].mean()), 2),
        "temperature_2m_min": round(float(weather_df["temperature_2m"].min()), 2),
        "temperature_2m_max": round(float(weather_df["temperature_2m"].max()), 2),
        "precipitation_total_mm": round(float(weather_df["precipitation"].sum()), 2),
    }

    report = {
        "metadata": {
            "notebook": "02_Data_Acquisition_and_Initial_Inspection",
            "section": "Section 3",
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "target_month": target_month,
            "ice_parquet": str(ice_parquet),
            "weather_parquet": str(weather_parquet),
            "primary_target": PRIMARY_TARGET,
            "primary_metric": "mae",
        },
        "api_request": {
            "url": OPEN_METEO_ARCHIVE_URL,
            "latitude": DEMO_LATITUDE,
            "longitude": DEMO_LONGITUDE,
            "start_date": start_date,
            "end_date": end_date,
            "timezone": DEMO_TIMEZONE,
            "hourly_variables": HOURLY_VARIABLES,
            "note": "Demo location only — all eva stations geocoded in Notebook 04",
        },
        "api_response_meta": {
            "returned_latitude": api_response.get("latitude"),
            "returned_longitude": api_response.get("longitude"),
            "timezone": api_response.get("timezone"),
            "hourly_units": api_response.get("hourly_units", {}),
        },
        "time_format_comparison": {
            "railway_raw": {
                "column": "departure_planned_time",
                "example": str(ice_with_dep["departure_planned_time"].iloc[0])
                if len(ice_with_dep)
                else None,
                "precision": "minute-level",
                "timezone": "naive in raw parquet — localized Europe/Berlin in Notebook 03",
            },
            "railway_merge_key": {
                "column": "departure_planned_hour / departure_planned_hour_naive",
                "example": str(ice_with_dep["departure_planned_hour"].iloc[0])
                if len(ice_with_dep)
                else None,
                "precision": "hour-level (floored)",
            },
            "weather_merge_key": {
                "column": "hourly.time",
                "example": str(weather_df["time"].iloc[0]),
                "precision": "hour-level",
                "timezone": DEMO_TIMEZONE,
            },
            "merge_rule": (
                "Join on eva + departure_planned_hour_naive == weather_time "
                "(after geocoding in Notebook 04)"
            ),
        },
        "alignment_stats": {
            "ice_rows_with_departure": int(len(ice_with_dep)),
            "unique_railway_hours": int(
                ice_with_dep["departure_planned_hour"].nunique()
            )
            if len(ice_with_dep)
            else 0,
            "unique_weather_hours": int(weather_df["time"].nunique()),
            "unique_eva_stations": int(ice_df["eva"].nunique()),
        },
        "weather_summary": weather_summary,
        "alignment_sample_rows": alignment_sample.to_dict(orient="records"),
        "documented_variables": list(weather_dict["hourly_variables"].keys()),
    }

    with report_path.open("w", encoding="utf-8") as f:
        json.dump(make_json_safe(report), f, indent=2, ensure_ascii=False)

    print(f"Weather hours : {len(weather_df):,}")
    print(
        f"Temp mean/min/max: {weather_summary['temperature_2m_mean']} / "
        f"{weather_summary['temperature_2m_min']} / "
        f"{weather_summary['temperature_2m_max']} °C"
    )
    print(f"Saved parquet : {weather_parquet}")
    print(f"Saved report  : {report_path}")
    print()
    print("Alignment sample (minute → hour):")
    print(alignment_sample.to_string(index=False))
    print()

    return {
        "month": target_month,
        "ice_rows": int(len(ice_df)),
        "weather_hours": int(len(weather_df)),
        "unique_eva": int(ice_df["eva"].nunique()),
        "temp_mean": weather_summary["temperature_2m_mean"],
        "precip_total_mm": weather_summary["precipitation_total_mm"],
        "weather_parquet": str(weather_parquet),
        "report": str(report_path),
    }


# =============================================================================
# RUN ALL MONTHS
# =============================================================================
print("Notebook 02 | Section 3 — Multi-month weather sample check")
print(f"Months : {', '.join(TARGET_MONTHS)}")
print(f"Target : {PRIMARY_TARGET} | Metric: MAE")
print()

month_results = [process_month_weather(m) for m in TARGET_MONTHS]

summary_path = REFERENCE_DIR / "weather_inspection_summary_multi_month.json"
summary = {
    "metadata": {
        "notebook": "02_Data_Acquisition_and_Initial_Inspection",
        "section": "Section 3",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "demo_location": {"latitude": DEMO_LATITUDE, "longitude": DEMO_LONGITUDE},
        "note": "Demo only — Notebook 04 does full per-station weather fetch",
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
    },
    "months": month_results,
}

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 3 COMPLETE — Weather sample summary")
print(sep)
print(f"{'Month':<10} {'ICE rows':>10} {'Wx hours':>10} {'Temp mean':>10}")
for r in month_results:
    print(
        f"{r['month']:<10} {r['ice_rows']:>10,} {r['weather_hours']:>10,} "
        f"{r['temp_mean']:>10.2f}"
    )
print()
print(f"Summary saved: {summary_path}")
print(sep)
print("Next: Section 4 — Notebook 02 summary / checklist")
print(sep)

Notebook 02 | Section 3 — Multi-month weather sample check
Months : 2024-07, 2024-08, 2024-09
Target : delay_in_min | Metric: MAE

WEATHER SAMPLE — 2024-07
ICE rows      : 157,886
Unique eva    : 95
Date range    : 2024-07-01 → 2024-07-31
Demo coords   : (52.525, 13.369) Berlin
Fetching Open-Meteo...
Weather hours : 744
Temp mean/min/max: 20.37 / 10.8 / 32.2 °C
Saved parquet : c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\weather_sample_2024-07.parquet
Saved report  : c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\weather_inspection_report_2024-07.json

Alignment sample (minute → hour):
     eva xml_station_name departure_planned_time departure_planned_hour  delay_in_min
08000049 Braunschweig Hbf    2024-07-01 00:01:00             2024-07-01             0
08000240        Mainz Hbf    2024-07-01 00:01:00             2024-07-01             0
08000261      München Hbf    2024-07-01 00:01:00             2024-07-01             1
08000098        Essen Hbf    

In [4]:
from pathlib import Path

processed = Path(r"C:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed")
reference = Path(r"C:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference")

for m in ["2024-07", "2024-08", "2024-09"]:
    wx = processed / f"weather_sample_{m}.parquet"
    rp = reference / f"weather_inspection_report_{m}.json"
    print(m, "weather:", wx.exists(), "report:", rp.exists())

2024-07 weather: True report: True
2024-08 weather: True report: True
2024-09 weather: True report: True


# Section 4: Notebook 02 Summary & Close-Out

**Why this section**  
Confirm all multi-month acquisition artifacts exist on disk before cleaning (Notebook 03).

**What we check**  
- `ice_raw_YYYY-MM.parquet` for Jul / Aug / Sep  
- Load + profile + weather sample reports  
- Multi-month summary JSONs  

**Output**  
`data/reference/notebook_02_summary.json`

**Next**  
Notebook 03 — clean & standardize each month (timezone, cancellations, merge keys).

In [5]:
# =============================================================================
# Notebook 02 | Section 4: Acquisition Summary & Close-Out (Multi-Month)
# =============================================================================
# Self-contained: verifies artifacts on disk; writes notebook_02_summary.json
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"
SUMMARY_PATH = REFERENCE_DIR / "notebook_02_summary.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def file_info(path: Path) -> dict:
    if not path.exists():
        return {"exists": False, "path": str(path), "size_mb": None}
    size_mb = round(path.stat().st_size / (1024 * 1024), 3)
    return {"exists": True, "path": str(path), "size_mb": size_mb}


config = load_json(CONFIG_PATH)
TARGET_MONTHS = config["scope"]["target_months"]
PRIMARY_TARGET = config["ml_tasks"]["primary"]["target"]
PRIMARY_METRIC = config["ml_tasks"]["primary"]["primary_metric"]

month_blocks = []
all_ok = True

print("=" * 72)
print("Notebook 02 | Section 4 — Multi-month artifact checklist")
print("=" * 72)
print(f"Months : {', '.join(TARGET_MONTHS)}")
print(f"Target : {PRIMARY_TARGET} | Metric: {PRIMARY_METRIC.upper()}")
print()

for month in TARGET_MONTHS:
    artifacts = {
        "ice_raw_parquet": PROCESSED_DIR / f"ice_raw_{month}.parquet",
        "ice_load_report": REFERENCE_DIR / f"ice_load_report_{month}.json",
        "ice_profile_report": REFERENCE_DIR / f"ice_profile_report_{month}.json",
        "weather_sample_parquet": PROCESSED_DIR / f"weather_sample_{month}.parquet",
        "weather_inspection_report": REFERENCE_DIR
        / f"weather_inspection_report_{month}.json",
    }

    print(f"--- {month} ---")
    details = {}
    month_ok = True
    for name, path in artifacts.items():
        info = file_info(path)
        details[name] = info
        status = "OK" if info["exists"] else "MISSING"
        if not info["exists"]:
            month_ok = False
            all_ok = False
        size = f"{info['size_mb']} MB" if info["exists"] else "-"
        print(f"  [{status}] {name:<28} {size}")

    ice_rows = None
    delay_mean = None
    stations = None
    if details["ice_raw_parquet"]["exists"]:
        ice_df = pd.read_parquet(artifacts["ice_raw_parquet"])
        ice_rows = int(len(ice_df))
        delay_mean = round(float(ice_df["delay_in_min"].mean()), 2)
        stations = int(ice_df["eva"].nunique())
        print(f"  ICE rows={ice_rows:,} | stations={stations} | delay_mean={delay_mean}")

    if details["ice_load_report"]["exists"] and ice_rows is not None:
        load_rep = load_json(artifacts["ice_load_report"])
        expected = load_rep["processing"]["ice_rows_saved"]
        if expected != ice_rows:
            print(f"  WARNING: load report rows {expected:,} != parquet {ice_rows:,}")
            month_ok = False
            all_ok = False

    month_blocks.append(
        {
            "month": month,
            "ok": month_ok,
            "ice_rows": ice_rows,
            "unique_eva_stations": stations,
            "delay_mean": delay_mean,
            "artifacts": details,
        }
    )
    print()

shared_artifacts = {
    "ice_load_summary_multi_month": REFERENCE_DIR
    / "ice_load_summary_multi_month.json",
    "ice_profile_summary_multi_month": REFERENCE_DIR
    / "ice_profile_summary_multi_month.json",
    "weather_inspection_summary_multi_month": REFERENCE_DIR
    / "weather_inspection_summary_multi_month.json",
}

print("--- Shared multi-month summaries ---")
shared_details = {}
for name, path in shared_artifacts.items():
    info = file_info(path)
    shared_details[name] = info
    status = "OK" if info["exists"] else "MISSING"
    if not info["exists"]:
        all_ok = False
    print(f"  [{status}] {name}")

print()

total_ice_rows = sum((m["ice_rows"] or 0) for m in month_blocks)

summary = {
    "metadata": {
        "notebook": "02_Data_Acquisition_and_Initial_Inspection",
        "section": "Section 4 — Close-Out",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "primary_target": PRIMARY_TARGET,
        "primary_metric": PRIMARY_METRIC,
        "authors": config["project"].get("authors", []),
    },
    "sections_completed": {
        "Section 1 — multi-month ICE load": all(
            (REFERENCE_DIR / f"ice_load_report_{m}.json").exists() for m in TARGET_MONTHS
        ),
        "Section 2 — multi-month ICE profile": all(
            (REFERENCE_DIR / f"ice_profile_report_{m}.json").exists()
            for m in TARGET_MONTHS
        ),
        "Section 3 — multi-month weather sample": all(
            (REFERENCE_DIR / f"weather_inspection_report_{m}.json").exists()
            for m in TARGET_MONTHS
        ),
        "Section 4 — close-out": True,
    },
    "months": month_blocks,
    "shared_artifacts": shared_details,
    "totals": {
        "ice_rows_all_months": total_ice_rows,
        "months_ok": sum(1 for m in month_blocks if m["ok"]),
        "months_total": len(TARGET_MONTHS),
    },
    "ready_for_notebook_03": all_ok,
    "next_notebook": {
        "name": "03_Data_Cleaning_and_Standardization",
        "tasks": [
            "Exclude canceled stops",
            "Localize timestamps to Europe/Berlin",
            "Create departure_planned_hour_naive merge key",
            "Save ice_cleaned_YYYY-MM.parquet for each month",
        ],
    },
}

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("NOTEBOOK 02 SUMMARY")
print(sep)
print(f"{'Month':<10} {'OK':<6} {'ICE rows':>12} {'Stations':>10} {'Mean delay':>11}")
for m in month_blocks:
    print(
        f"{m['month']:<10} {str(m['ok']):<6} "
        f"{(m['ice_rows'] or 0):>12,} "
        f"{(m['unique_eva_stations'] or 0):>10} "
        f"{(m['delay_mean'] if m['delay_mean'] is not None else float('nan')):>11}"
    )
print()
print(f"Total ICE rows        : {total_ice_rows:,}")
print(f"Ready for Notebook 03 : {all_ok}")
print(f"Summary saved         : {SUMMARY_PATH}")
print(sep)
if all_ok:
    print("Notebook 02 complete. Next: Notebook 03 — multi-month cleaning")
else:
    print("Fix MISSING artifacts above, then re-run Sections 1–3 as needed.")
print(sep)

Notebook 02 | Section 4 — Multi-month artifact checklist
Months : 2024-07, 2024-08, 2024-09
Target : delay_in_min | Metric: MAE

--- 2024-07 ---
  [OK] ice_raw_parquet              5.243 MB
  [OK] ice_load_report              0.002 MB
  [OK] ice_profile_report           0.004 MB
  [OK] weather_sample_parquet       0.018 MB
  [OK] weather_inspection_report    0.004 MB
  ICE rows=157,886 | stations=95 | delay_mean=10.9

--- 2024-08 ---
  [OK] ice_raw_parquet              5.054 MB
  [OK] ice_load_report              0.002 MB
  [OK] ice_profile_report           0.005 MB
  [OK] weather_sample_parquet       0.018 MB
  [OK] weather_inspection_report    0.004 MB
  ICE rows=150,289 | stations=98 | delay_mean=11.91

--- 2024-09 ---
  [OK] ice_raw_parquet              5.04 MB
  [OK] ice_load_report              0.002 MB
  [OK] ice_profile_report           0.005 MB
  [OK] weather_sample_parquet       0.018 MB
  [OK] weather_inspection_report    0.004 MB
  ICE rows=149,218 | stations=98 | delay_mea